# 03_ngram_preprocessing_smoothing.ipynb

## N-gram-Specific Preprocessing and Smoothing

**Starts after Notebook 01.** Do not repeat translation, translation validation, clinical normalization, leakage removal, generic EDA, or splitting.

### Input
- `data/splits/train.csv`
- `data/splits/validation.csv`
- `data/splits/test.csv`

Expected columns: `id`, `raw_text`, `text`, `label`

### This notebook's responsibility
1. Freeze the shared tokenizer.
2. Build vocabulary from training data only.
3. Map rare/unseen words to `<UNK>`.
4. Build class-specific unigram/bigram/trigram statistics.
5. Prepare smoothed probabilities:
   - Unigram: add-alpha base distribution.
   - Bigram: interpolated Kneser-Ney.
   - Trigram: interpolated Kneser-Ney with bigram backoff.
6. Export artifacts and the shared scorer for Aswathy.

### Hand-off interface

```python
scorer.score(tokens, class_label, order)
```

returns:

```text
log P(tokens | class_label)
```

The next notebook performs MAP classification by adding `log P(class)` and selecting the class with the highest score.


In [ ]:
# 1. Imports
import os
import re
import json
import math
import random
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import joblib

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 3. Configuration

In [ ]:
# EDIT ONLY BASE_DIR IF YOUR DRIVE PATH DIFFERS

BASE_DIR = '/content/drive/MyDrive/NLP_Research_Project'

ID_COLUMN = 'id'
TEXT_COLUMN = 'text'
LABEL_COLUMN = 'label'

# Tokens occurring fewer than this in TRAINING data become <UNK>.
MIN_TOKEN_FREQ = 2

# Standalone unigram smoothing.
UNIGRAM_ALPHA = 1.0

SPLITS_DIR = os.path.join(BASE_DIR, 'data', 'splits')
TRAIN_PATH = os.path.join(SPLITS_DIR, 'train.csv')
VAL_PATH = os.path.join(SPLITS_DIR, 'validation.csv')
TEST_PATH = os.path.join(SPLITS_DIR, 'test.csv')

ARTIFACT_DIR = os.path.join(BASE_DIR, 'artifacts', 'ngram')
AUDIT_DIR = os.path.join(BASE_DIR, 'results', 'audit')
SRC_DIR = os.path.join(BASE_DIR, 'src')

for d in [ARTIFACT_DIR, AUDIT_DIR, SRC_DIR]:
    os.makedirs(d, exist_ok=True)

ARTIFACT_PATH = os.path.join(ARTIFACT_DIR, 'ngram_language_models.joblib')
METADATA_PATH = os.path.join(ARTIFACT_DIR, 'ngram_metadata.json')
VOCABULARY_PATH = os.path.join(ARTIFACT_DIR, 'shared_vocabulary.json')
TOKENIZER_AUDIT_PATH = os.path.join(AUDIT_DIR, 'ngram_tokenization_audit.json')
MODULE_PATH = os.path.join(SRC_DIR, 'ngram_utils.py')

print(TRAIN_PATH)
print(VAL_PATH)
print(TEST_PATH)


/content/drive/MyDrive/NLP_Research_Project/data/splits/train.csv
/content/drive/MyDrive/NLP_Research_Project/data/splits/validation.csv
/content/drive/MyDrive/NLP_Research_Project/data/splits/test.csv


## 4. Load the Existing Processed Splits

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

required = {ID_COLUMN, TEXT_COLUMN, LABEL_COLUMN}
for name, df in {'train': train_df, 'validation': val_df, 'test': test_df}.items():
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'{name} missing columns: {missing}')
    if df[TEXT_COLUMN].isna().any() or df[LABEL_COLUMN].isna().any():
        raise ValueError(f'{name} contains missing text or labels')

print('Train:', train_df.shape)
print('Validation:', val_df.shape)
print('Test:', test_df.shape)
print('Classes:', sorted(train_df[LABEL_COLUMN].unique()))


Train: (6425, 4)
Validation: (1377, 4)
Test: (1377, 4)
Classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]


## 5. Shared Clinical Tokenizer

The input is already clinically normalized by Notebook 01. We do **not** remove stopwords, stem, or lemmatize.

This preserves negation and clinical distinctions such as `no`, `without`, laterality, measurements, hyphenated terms, and slash-containing tokens.

The tokenizer is deterministic and will be exported for the classifier notebook.


In [ ]:
TOKEN_PATTERN = re.compile(r"[a-z0-9]+(?:[./-][a-z0-9]+)*")

def tokenize(text):
    text = '' if pd.isna(text) else str(text).lower().strip()
    return TOKEN_PATTERN.findall(text)

sample = train_df[TEXT_COLUMN].iloc[0]
print(sample)
print(tokenize(sample)[:80])


clinical indication tracking. family history of breast cancer (sister). findings heterogeneously dense breasts, which may hide small nodules. sparse calcifications. no suspicious grouped calcifications were observed. bilateral axillary lymph nodes with normal appearance. comparative analysis regarding the examination, no significant changes were observed
['clinical', 'indication', 'tracking', 'family', 'history', 'of', 'breast', 'cancer', 'sister', 'findings', 'heterogeneously', 'dense', 'breasts', 'which', 'may', 'hide', 'small', 'nodules', 'sparse', 'calcifications', 'no', 'suspicious', 'grouped', 'calcifications', 'were', 'observed', 'bilateral', 'axillary', 'lymph', 'nodes', 'with', 'normal', 'appearance', 'comparative', 'analysis', 'regarding', 'the', 'examination', 'no', 'significant', 'changes', 'were', 'observed']


## 6. Shared Vocabulary and `<UNK>` Policy

Vocabulary is built from **training data only**.

- frequency >= `MIN_TOKEN_FREQ` → keep token
- frequency < `MIN_TOKEN_FREQ` → `<UNK>`
- any unseen validation/test token → `<UNK>`

This prevents vocabulary leakage from validation or test data.


In [ ]:
training_counts = Counter()

for text in train_df[TEXT_COLUMN]:
    training_counts.update(tokenize(text))

shared_vocabulary = {
    token for token, count in training_counts.items()
    if count >= MIN_TOKEN_FREQ
}

def apply_vocabulary(tokens):
    return [t if t in shared_vocabulary else '<UNK>' for t in tokens]

for df in [train_df, val_df, test_df]:
    df['ngram_tokens'] = df[TEXT_COLUMN].apply(
        lambda x: apply_vocabulary(tokenize(x))
    )

def unk_stats(df):
    total = sum(len(x) for x in df['ngram_tokens'])
    unk = sum(t == '<UNK>' for seq in df['ngram_tokens'] for t in seq)
    return {
        'total_tokens': int(total),
        'unk_tokens': int(unk),
        'unk_rate': float(unk / total) if total else 0.0
    }

audit = {
    'min_token_frequency': MIN_TOKEN_FREQ,
    'vocabulary_size': len(shared_vocabulary),
    'train': unk_stats(train_df),
    'validation': unk_stats(val_df),
    'test': unk_stats(test_df)
}

with open(TOKENIZER_AUDIT_PATH, 'w') as f:
    json.dump(audit, f, indent=2)

audit


{'min_token_frequency': 2,
 'vocabulary_size': 1592,
 'train': {'total_tokens': 338448,
  'unk_tokens': 867,
  'unk_rate': 0.002561693376825982},
 'validation': {'total_tokens': 72246,
  'unk_tokens': 391,
  'unk_rate': 0.0054120643357417715},
 'test': {'total_tokens': 71444,
  'unk_tokens': 350,
  'unk_rate': 0.00489894182856503}}

## 7. Report Boundary Handling

Each report is treated as one sequence.

- Unigram: report tokens
- Bigram: `<BOS> tokens <EOS>`
- Trigram: `<BOS> <BOS> tokens <EOS>`

Boundary markers are internal and Aswathy does not add them manually.


In [ ]:
def add_boundaries(tokens, order):
    tokens = list(tokens)
    if order == 1:
        return tokens
    if order == 2:
        return ['<BOS>'] + tokens + ['<EOS>']
    if order == 3:
        return ['<BOS>', '<BOS>'] + tokens + ['<EOS>']
    raise ValueError('order must be 1, 2, or 3')


## 8. Build Per-Class N-gram Statistics

For each BI-RADS class, we build a separate class-conditional statistical language model.

The same report can later be scored under all seven class models.


In [ ]:
def estimate_discount(counts):
    fof = Counter(counts.values())
    n1 = fof.get(1, 0)
    n2 = fof.get(2, 0)

    if n1 == 0 or n2 == 0:
        return 0.75

    D = n1 / (n1 + 2 * n2)
    return float(min(max(D, 0.1), 0.99))


def build_class_statistics(token_sequences):
    unigram_counts = Counter()
    bigram_counts = Counter()
    bigram_context_counts = Counter()

    trigram_counts = Counter()
    trigram_context_counts = Counter()

    for tokens in token_sequences:
        tokens = list(tokens)
        unigram_counts.update(tokens)

        seq2 = add_boundaries(tokens, 2)
        for i in range(1, len(seq2)):
            h, w = seq2[i-1], seq2[i]
            bigram_counts[(h, w)] += 1
            bigram_context_counts[h] += 1

        seq3 = add_boundaries(tokens, 3)
        for i in range(2, len(seq3)):
            h1, h2, w = seq3[i-2], seq3[i-1], seq3[i]
            trigram_counts[(h1, h2, w)] += 1
            trigram_context_counts[(h1, h2)] += 1

    bigram_unique_continuations = Counter()
    for h, w in bigram_counts:
        bigram_unique_continuations[h] += 1

    trigram_unique_continuations = Counter()
    for h1, h2, w in trigram_counts:
        trigram_unique_continuations[(h1, h2)] += 1

    predecessor_sets = defaultdict(set)
    for h, w in bigram_counts:
        predecessor_sets[w].add(h)

    continuation_counts = {
        w: len(histories)
        for w, histories in predecessor_sets.items()
    }

    return {
        'unigram_counts': unigram_counts,
        'unigram_total': int(sum(unigram_counts.values())),

        'bigram_counts': bigram_counts,
        'bigram_context_counts': bigram_context_counts,
        'bigram_unique_continuations': bigram_unique_continuations,
        'continuation_counts': continuation_counts,
        'total_bigram_types': int(len(bigram_counts)),

        'trigram_counts': trigram_counts,
        'trigram_context_counts': trigram_context_counts,
        'trigram_unique_continuations': trigram_unique_continuations,

        'bigram_discount': estimate_discount(bigram_counts),
        'trigram_discount': estimate_discount(trigram_counts),
    }


In [ ]:
classes = sorted(train_df[LABEL_COLUMN].unique())
class_models = {}

for class_label in classes:
    sequences = train_df.loc[
        train_df[LABEL_COLUMN] == class_label,
        'ngram_tokens'
    ].tolist()

    class_models[int(class_label)] = build_class_statistics(sequences)

    model = class_models[int(class_label)]
    print(
        f'Class {class_label}: '
        f'{len(sequences)} reports | '
        f'{model["unigram_total"]} tokens | '
        f'{len(model["bigram_counts"])} bigram types | '
        f'{len(model["trigram_counts"])} trigram types'
    )


Class 0: 423 reports | 25473 tokens | 2514 bigram types | 4010 trigram types
Class 1: 176 reports | 6705 tokens | 847 bigram types | 1110 trigram types
Class 2: 5126 reports | 256683 tokens | 10110 bigram types | 20236 trigram types
Class 3: 498 reports | 35114 tokens | 4219 bigram types | 7355 trigram types
Class 4: 150 reports | 10517 tokens | 2252 bigram types | 3528 trigram types
Class 5: 20 reports | 1428 tokens | 611 bigram types | 806 trigram types
Class 6: 32 reports | 2528 tokens | 1185 bigram types | 1570 trigram types


## 9. Training-Only Class Priors

These are exported for Aswathy's MAP classification stage.

This notebook does not make final class predictions.


In [ ]:
class_counts = train_df[LABEL_COLUMN].value_counts().sort_index()

class_priors = {
    int(label): float(count / len(train_df))
    for label, count in class_counts.items()
}

class_priors


{0: 0.06583657587548639,
 1: 0.027392996108949415,
 2: 0.7978210116731518,
 3: 0.07750972762645915,
 4: 0.023346303501945526,
 5: 0.0031128404669260703,
 6: 0.004980544747081712}

## 10. Export Shared Artifacts

The artifact contains all learned count/smoothing statistics.

The source module provides the common interface:

```python
from ngram_utils import tokenize, NGramScorer

scorer = NGramScorer(ARTIFACT_PATH)
scorer.score(tokens, class_label, order)
```


In [ ]:
module_code = 'import re\nimport math\nimport joblib\n\nTOKEN_PATTERN = re.compile(r"[a-z0-9]+(?:[./-][a-z0-9]+)*")\nEPSILON = 1e-300\n\ndef tokenize(text):\n    text = "" if text is None else str(text).lower().strip()\n    return TOKEN_PATTERN.findall(text)\n\nclass NGramScorer:\n    def __init__(self, artifact_path):\n        self.artifacts = joblib.load(artifact_path)\n        self.vocabulary = set(self.artifacts["vocabulary"])\n        self.class_models = {int(k): v for k, v in self.artifacts["class_models"].items()}\n        self.class_priors = {int(k): float(v) for k, v in self.artifacts["class_priors"].items()}\n        self.unigram_alpha = float(self.artifacts["unigram_alpha"])\n        self.vocabulary_size = int(self.artifacts["vocabulary_size"])\n\n    def map_tokens(self, tokens):\n        return [t if t in self.vocabulary else "<UNK>" for t in tokens]\n\n    def _unigram_probability(self, model, word):\n        V = self.vocabulary_size + 1\n        return (model["unigram_counts"].get(word, 0) + self.unigram_alpha) / (\n            model["unigram_total"] + self.unigram_alpha * V\n        )\n\n    def _continuation_probability(self, model, word):\n        total_types = model["total_bigram_types"]\n        if total_types == 0:\n            return self._unigram_probability(model, word)\n        p = model["continuation_counts"].get(word, 0) / total_types\n        return p if p > 0 else self._unigram_probability(model, word)\n\n    def _bigram_probability(self, model, history, word):\n        context_count = model["bigram_context_counts"].get(history, 0)\n        if context_count == 0:\n            return self._continuation_probability(model, word)\n\n        count = model["bigram_counts"].get((history, word), 0)\n        D = model["bigram_discount"]\n        first = max(count - D, 0.0) / context_count\n        n1plus = model["bigram_unique_continuations"].get(history, 0)\n        lam = D * n1plus / context_count\n        return first + lam * self._continuation_probability(model, word)\n\n    def _trigram_probability(self, model, h1, h2, word):\n        context = (h1, h2)\n        context_count = model["trigram_context_counts"].get(context, 0)\n        if context_count == 0:\n            return self._bigram_probability(model, h2, word)\n\n        count = model["trigram_counts"].get((h1, h2, word), 0)\n        D = model["trigram_discount"]\n        first = max(count - D, 0.0) / context_count\n        n1plus = model["trigram_unique_continuations"].get(context, 0)\n        lam = D * n1plus / context_count\n        return first + lam * self._bigram_probability(model, h2, word)\n\n    def score(self, tokens, class_label, order):\n        if order not in (1, 2, 3):\n            raise ValueError("order must be 1, 2, or 3")\n\n        label = int(class_label)\n        model = self.class_models[label]\n        tokens = self.map_tokens(tokens)\n        ll = 0.0\n\n        if order == 1:\n            for word in tokens:\n                ll += math.log(max(self._unigram_probability(model, word), EPSILON))\n            return ll\n\n        if order == 2:\n            seq = ["<BOS>"] + tokens + ["<EOS>"]\n            for i in range(1, len(seq)):\n                p = self._bigram_probability(model, seq[i-1], seq[i])\n                ll += math.log(max(p, EPSILON))\n            return ll\n\n        seq = ["<BOS>", "<BOS>"] + tokens + ["<EOS>"]\n        for i in range(2, len(seq)):\n            p = self._trigram_probability(model, seq[i-2], seq[i-1], seq[i])\n            ll += math.log(max(p, EPSILON))\n        return ll\n\n    def log_prior(self, class_label):\n        return math.log(max(self.class_priors[int(class_label)], EPSILON))\n'
with open(MODULE_PATH, 'w', encoding='utf-8') as f:
    f.write(module_code)

print('Written:', MODULE_PATH)


Written: /content/drive/MyDrive/NLP_Research_Project/src/ngram_utils.py


In [ ]:
artifacts = {
    'vocabulary': sorted(shared_vocabulary),
    'vocabulary_size': len(shared_vocabulary),
    'min_token_frequency': MIN_TOKEN_FREQ,
    'unigram_alpha': UNIGRAM_ALPHA,
    'classes': [int(x) for x in classes],
    'class_priors': class_priors,
    'class_models': class_models,
}

joblib.dump(artifacts, ARTIFACT_PATH)

with open(VOCABULARY_PATH, 'w') as f:
    json.dump(sorted(shared_vocabulary), f, indent=2)

metadata = {
    'input': 'data/splits/train.csv',
    'validation': 'data/splits/validation.csv',
    'test': 'data/splits/test.csv',
    'tokenizer_pattern': TOKEN_PATTERN.pattern,
    'min_token_frequency': MIN_TOKEN_FREQ,
    'unknown_token': '<UNK>',
    'boundary_tokens': ['<BOS>', '<EOS>'],
    'unigram': 'add-alpha, alpha=' + str(UNIGRAM_ALPHA),
    'bigram': 'interpolated Kneser-Ney',
    'trigram': 'interpolated Kneser-Ney with bigram backoff',
    'classes': [int(x) for x in classes]
}

with open(METADATA_PATH, 'w') as f:
    json.dump(metadata, f, indent=2)

print('Saved:')
print(ARTIFACT_PATH)
print(VOCABULARY_PATH)
print(METADATA_PATH)
print(MODULE_PATH)


Saved:
/content/drive/MyDrive/NLP_Research_Project/artifacts/ngram/ngram_language_models.joblib
/content/drive/MyDrive/NLP_Research_Project/artifacts/ngram/shared_vocabulary.json
/content/drive/MyDrive/NLP_Research_Project/artifacts/ngram/ngram_metadata.json
/content/drive/MyDrive/NLP_Research_Project/src/ngram_utils.py


## 11. Sanity-Test the Shared Interface

In [ ]:
import sys
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

from ngram_utils import tokenize as shared_tokenize, NGramScorer

scorer = NGramScorer(ARTIFACT_PATH)

sample_report = val_df[TEXT_COLUMN].iloc[0]
sample_tokens = shared_tokenize(sample_report)

print('Tokens:', sample_tokens[:30])

for order in (1, 2, 3):
    print(f'\nOrder {order}')
    for class_label in classes:
        s = scorer.score(sample_tokens, int(class_label), order)
        print(f'Class {class_label}: {s:.4f}')


Tokens: ['clinical', 'indication', 'tracking', 'findings', 'partially', 'liposubstituted', 'breasts', 'no', 'grouped', 'pleomorphic', 'microcalcifications', 'were', 'observed', 'sparse', 'calcifications', 'bilateral', 'intramammary', 'lymph', 'nodes', 'the', 'axillary', 'regions', 'do', 'not', 'present', 'significant', 'changes', 'comparative', 'analysis', 'regarding']

Order 1
Class 0: -178.4397
Class 1: -170.0428
Class 2: -159.9585
Class 3: -175.5788
Class 4: -186.3243
Class 5: -211.2003
Class 6: -204.4025

Order 2
Class 0: -52.1101
Class 1: -71.9052
Class 2: -40.9590
Class 3: -44.5344
Class 4: -62.6041
Class 5: -125.7460
Class 6: -85.5692

Order 3
Class 0: -45.4982
Class 1: -74.0251
Class 2: -26.9885
Class 3: -30.0046
Class 4: -52.5020
Class 5: -129.2434
Class 6: -78.2419


## 12. Verify `<UNK>` Handling

In [ ]:
unseen_tokens = ['completelyunseenclinicaltoken', 'suspicious', 'lesion']

for order in (1, 2, 3):
    value = scorer.score(unseen_tokens, int(classes[0]), order)
    assert math.isfinite(value), f'Non-finite score at order {order}'

print('PASS: unseen words are safely handled through <UNK>.')


PASS: unseen words are safely handled through <UNK>.


# Final Handoff

Share these outputs:

```text
artifacts/ngram/ngram_language_models.joblib
artifacts/ngram/ngram_metadata.json
artifacts/ngram/shared_vocabulary.json
src/ngram_utils.py
```

Her classification notebook should:

1. Load the scorer.
2. Tokenize a report using `tokenize(text)`.
3. For each class 0–6 calculate:

```text
scorer.score(tokens, class_label, order) + scorer.log_prior(class_label)
```

4. Select the class with maximum score.
5. Repeat separately for `order = 1`, `2`, and `3`.
6. Evaluate validation/test predictions.
